# NB04 — Author Frozen-Encoder Transfer Reproduction (Table 9/10 Head)

Reproduces the Ben-Sarc transfer-learning family (Table 9): three **frozen** transformer encoders — `sagorsarker/bangla-bert-base` ("Bangla BERT"), `bert-base-multilingual-cased` ("M-BERT"), and `l3cube-pune/bengali-bert` (substitute for the now-offline Indic-Transformers Bengali BERT) — each topped with the **exact classification head from Table 10**. Runs on **RunPod (GPU)**.

**How fidelity is preserved:** the encoder is frozen and never updated, so we precompute its `[CLS]` embeddings once and train only the head on top — mathematically identical to training a head on a frozen encoder, just faster.

**Head (Table 10):** 7 hidden Dense layers `[512,256,128,64,32,16,8]` (tanh, dropout 0.1) + 1 output Dense = **8 Dense layers total**, Adam @ 1e-4, batch 8, 30 epochs. `REFERENCE_TARGETS` is pre-filled from Table 9 (Ben-Sarc binary).

Defense-in-depth: every split is re-checked for train/val/test leakage at load (hard fail on overlap).

In [1]:
# Ensure dependencies (fast no-op if already on the pod)
import importlib, subprocess, sys
def ensure(pip_name, import_name=None):
    try: importlib.import_module(import_name or pip_name)
    except ImportError:
        print("installing", pip_name); subprocess.run([sys.executable,"-m","pip","install","-q",pip_name], check=False)
for p,i in [("transformers","transformers"),("torch","torch"),("scikit-learn","sklearn"),
            ("pandas","pandas"),("numpy","numpy")]:
    ensure(p,i)
print("deps ok")

deps ok


In [2]:
import os, json, time, random, re, unicodedata, gc
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             confusion_matrix, classification_report)
import transformers
print("torch:", torch.__version__, "| transformers:", transformers.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", DEVICE)

torch: 2.8.0+cu128 | transformers: 4.40.0
device: cuda


In [3]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print("seed set:", SEED)

seed set: 42


In [4]:
def find_repo_root():
    cands = [Path("/workspace/Sarcasm_detection"), Path.cwd()] + list(Path.cwd().resolve().parents)
    for c in cands:
        if (c / "01_data").exists(): return c.resolve()
    raise RuntimeError("Could not locate repo root (no 01_data dir).")
HEAD_DEVICE = "cpu"
ROOT     = find_repo_root()
SPLITS   = ROOT / "01_data" / "interim" / "splits"
CKPT     = ROOT / "03_checkpoints"
TABLES   = ROOT / "04_outputs" / "tables"
PRED     = ROOT / "04_outputs" / "predictions"
REPORTS  = ROOT / "04_outputs" / "reports"
FIGURES  = ROOT / "04_outputs" / "figures"
for p in [CKPT, TABLES, PRED, REPORTS, FIGURES]: p.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "| SPLITS exist:", SPLITS.exists())
assert SPLITS.exists(), f"Missing splits at {SPLITS}. Run NB01 first." 

ROOT: /workspace/Sarcasm_detection | SPLITS exist: True


In [5]:
# ---- normalized key + leakage guard ----
_ZW = dict.fromkeys(map(ord, ["\u200b","\u200c","\u200d","\ufeff"]), None)
def norm_key(s):
    if not isinstance(s, str): s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFC", s).translate(_ZW)
    return re.sub(r"\s+", " ", s).strip().casefold()

def assert_no_internal_leakage(tr, va, te, col="text"):
    s_tr={norm_key(x) for x in tr[col]}; s_va={norm_key(x) for x in va[col]}; s_te={norm_key(x) for x in te[col]}
    a,b,c = len(s_tr&s_va), len(s_tr&s_te), len(s_va&s_te)
    assert a==0, f"LEAK train/val={a}"; assert b==0, f"LEAK train/test={b}"; assert c==0, f"LEAK val/test={c}"
    return {"train_val":a,"train_test":b,"val_test":c}

def metrics_from_preds(y_true, logits, num_labels):
    y_true=np.asarray(y_true); preds=np.asarray(logits).argmax(-1)
    m={"accuracy":float(accuracy_score(y_true,preds)),
       "macro_f1":float(f1_score(y_true,preds,average="macro",zero_division=0)),
       "weighted_f1":float(f1_score(y_true,preds,average="weighted",zero_division=0)),
       "macro_precision":float(precision_score(y_true,preds,average="macro",zero_division=0)),
       "macro_recall":float(recall_score(y_true,preds,average="macro",zero_division=0))}
    if num_labels==2:
        m["binary_f1"]=float(f1_score(y_true,preds,average="binary",pos_label=1,zero_division=0))
    return m, preds

def softmax_np(x):
    x=np.asarray(x,dtype=np.float64); x=x-x.max(1,keepdims=True); e=np.exp(x); return e/e.sum(1,keepdims=True)

def find_label_col(df):
    for c in ["label_binary","label_ternary","label","target","class"]:
        if c in df.columns: return c
    raise ValueError(f"No label column in {list(df.columns)}")

def load_task(task):
    def rd(split):
        df=pd.read_csv(SPLITS/f"{task}_{split}.csv"); lc=find_label_col(df)
        df=df[["text",lc]].dropna(subset=["text"]).copy()
        df["text"]=df["text"].astype(str); df[lc]=df[lc].astype(int)
        return df, lc
    tr,lc=rd("train"); va,_=rd("val"); te,_=rd("test")
    leak=assert_no_internal_leakage(tr,va,te)
    return tr,va,te,lc,int(tr[lc].nunique()),leak

In [6]:
# ============================ CONFIG ============================
# Frozen-encoder transfer family (Table 9 of the Ben-Sarc reference paper, Experiment III).
# 'bengali-bert' substitutes the offline Indic-Transformers Bengali-BERT (is_substitute flagged).
ENCODERS = {
    "sagorsarker-bb": {"name": "sagorsarker/bangla-bert-base",     "is_substitute": False},  # paper's "Bangla BERT"
    "mbert":          {"name": "bert-base-multilingual-cased",     "is_substitute": False},  # paper's "M-BERT"
    "bengali-bert":   {"name": "l3cube-pune/bengali-bert",         "is_substitute": True},   # sub for Indic-Transformers Bengali BERT
}

# Reproduction target(s). Reference gap is computed only where REFERENCE_TARGETS is filled.
TASKS = ["ben_sarc_binary", "banglasarc_binary", "banglasarc3_binary"]

# ----- Lora et al. classification head (Table 10): EXACTLY 7 hidden Dense layers + 1 output -----
HEAD_HIDDEN_UNITS = [512, 256, 128, 64, 32, 16, 8]   # 7 hidden Dense (Table 10) -> + 1 output Dense = 8 total
HEAD_ACTIVATION   = "tanh"                            # Table 10
HEAD_DROPOUT      = 0.1                               # Table 10
POOLING           = "cls"                             # "cls" or "mean"

MAX_LENGTH = 128
EMB_BATCH  = 64          # batch for frozen-encoder feature extraction (does not affect fidelity)
HEAD_LR    = 1e-4        # Adam, lr=0.0001 (Table 10)
HEAD_EPOCHS = 30         # Table 10
HEAD_PATIENCE = 30       # >= HEAD_EPOCHS -> effectively disables early stop, runs the full 30 (Table 10)
HEAD_BATCH = 64           # Table 10
USE_FP16_ENCODER = torch.cuda.is_available()

# Table 9 (Ben-Sarc, Experiment III). F1 in the paper is reported per-class avg on a balanced
# binary set, so macro and weighted F1 coincide; used directly as reference_f1 here.
REFERENCE_TARGETS = {
    ("mbert",          "ben_sarc_binary"): {"reference_accuracy": 0.6600, "reference_f1": 0.6547},
    ("bengali-bert",   "ben_sarc_binary"): {"reference_accuracy": 0.7505, "reference_f1": 0.7547},
    ("sagorsarker-bb", "ben_sarc_binary"): {"reference_accuracy": 0.6800, "reference_f1": 0.6641},
}

assert len(HEAD_HIDDEN_UNITS) == 7, "Paper specifies 7 hidden Dense layers (Table 10)."
print(f"Head -> {len(HEAD_HIDDEN_UNITS)} hidden Dense {HEAD_HIDDEN_UNITS} + 1 output "
      f"({HEAD_ACTIVATION}, dropout={HEAD_DROPOUT}) | pooling={POOLING}")
print("encoders:", list(ENCODERS), "| tasks:", TASKS)

Head -> 7 hidden Dense [512, 256, 128, 64, 32, 16, 8] + 1 output (tanh, dropout=0.1) | pooling=cls
encoders: ['sagorsarker-bb', 'mbert', 'bengali-bert'] | tasks: ['ben_sarc_binary', 'banglasarc_binary', 'banglasarc3_binary']


In [7]:
class DenseHead(nn.Module):
    """Lora et al. Table 10 head: 7 hidden Dense (activation+dropout) + 1 output Dense = 8 total."""
    _ACT = {"relu": nn.ReLU, "gelu": nn.GELU, "tanh": nn.Tanh}
    def __init__(self, in_dim, hidden_units, num_labels, dropout=0.1, activation="tanh"):
        super().__init__()
        act = self._ACT[activation]
        layers, prev = [], in_dim
        for h in hidden_units:                       # 7 hidden Dense
            layers += [nn.Linear(prev, h), act(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, num_labels)]      # 8th Dense = output
        self.net = nn.Sequential(*layers)
        self.n_dense = sum(isinstance(m, nn.Linear) for m in self.net)
    def forward(self, x): return self.net(x)

@torch.no_grad()
def extract_embeddings(encoder, tokenizer, texts, pooling="cls"):
    encoder.eval()
    chunks = []
    autocast = (USE_FP16_ENCODER and DEVICE == "cuda")
    for i in range(0, len(texts), EMB_BATCH):
        bt = list(texts[i:i+EMB_BATCH])
        enc = tokenizer(bt, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=autocast):
            out = encoder(**enc)
        last = out.last_hidden_state.float()
        if pooling == "cls":
            v = last[:, 0]
        else:
            m = enc["attention_mask"].unsqueeze(-1).float()
            v = (last * m).sum(1) / m.sum(1).clamp(min=1e-9)
        chunks.append(v.cpu())
    return torch.cat(chunks).numpy()

def train_head(Xtr, ytr, Xva, yva, in_dim, num_labels):
    torch.manual_seed(SEED); np.random.seed(SEED)
    dev = DEVICE
    head = DenseHead(in_dim, HEAD_HIDDEN_UNITS, num_labels, HEAD_DROPOUT, HEAD_ACTIVATION).to(dev)
    assert head.n_dense == 8, f"Head has {head.n_dense} Dense layers, expected 8"
    opt = torch.optim.Adam(head.parameters(), lr=HEAD_LR)
    lossf = nn.CrossEntropyLoss()
    # ALL features resident on-device ONCE -> no per-step host<->device copies (this is the fix)
    Xtr_t = torch.as_tensor(Xtr, dtype=torch.float32, device=dev)
    ytr_t = torch.as_tensor(ytr, dtype=torch.long,    device=dev)
    Xva_t = torch.as_tensor(Xva, dtype=torch.float32, device=dev)
    yva_np = np.asarray(yva)
    n = len(Xtr_t)
    best_f1, best_state, bad = -1.0, None, 0
    t0 = time.time()
    for ep in range(HEAD_EPOCHS):
        head.train()
        perm = torch.randperm(n, device=dev)          # permutation on-device too
        for s in range(0, n, HEAD_BATCH):
            idx = perm[s:s+HEAD_BATCH]
            opt.zero_grad(set_to_none=True)
            loss = lossf(head(Xtr_t[idx]), ytr_t[idx])
            loss.backward(); opt.step()
        head.eval()
        with torch.no_grad():
            vpred = head(Xva_t).argmax(-1).cpu().numpy()
        f1 = f1_score(yva_np, vpred, average="macro", zero_division=0)
        if f1 > best_f1 + 1e-5:
            best_f1, best_state, bad = f1, {k: v.detach().clone() for k, v in head.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= HEAD_PATIENCE: break
    if best_state is not None: head.load_state_dict(best_state)
    print(f"    head: {ep+1} epochs | best_val_f1={best_f1:.4f} | {time.time()-t0:.1f}s")
    return head, best_f1, ep + 1

@torch.no_grad()
def head_logits(head, X):
    head.eval()
    return head(torch.as_tensor(X, dtype=torch.float32, device=DEVICE)).cpu().numpy()

In [8]:
results, confusions = [], {}
SUMMARY = TABLES / "04_frozen_transfer_results.csv"
done = set()
if SUMMARY.exists():
    prev = pd.read_csv(SUMMARY); results = prev.to_dict("records")
    done = set(zip(prev["encoder"], prev["task"])); print("resuming, done:", len(done))

for task in TASKS:
    tr, va, te, lc, num_labels, leak = load_task(task)
    print("\n" + "="*78 + f"\nTASK: {task}  (labels={num_labels}, leak={leak})\n" + "="*78)
    for etag, einfo in ENCODERS.items():
        if (etag, task) in done:
            print(f"  skip {etag} x {task}"); continue
        ename = einfo["name"]
        print(f"\n[{etag}] {ename}  -> extracting frozen embeddings ...")
        t0 = time.time()
        tok = AutoTokenizer.from_pretrained(ename)
        enc = AutoModel.from_pretrained(ename).to(DEVICE)
        Xtr = extract_embeddings(enc, tok, tr["text"].tolist(), POOLING)
        Xva = extract_embeddings(enc, tok, va["text"].tolist(), POOLING)
        Xte = extract_embeddings(enc, tok, te["text"].tolist(), POOLING)
        in_dim = Xtr.shape[1]
        del enc; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        head, best_val_f1, ep_ran = train_head(Xtr, tr[lc].values, Xva, va[lc].values, in_dim, num_labels)
        test_logits = head_logits(head, Xte)
        m, preds = metrics_from_preds(te[lc].values, test_logits, num_labels)
        secs = round(time.time() - t0, 1)
        print(f"   -> test macro-F1={m['macro_f1']:.4f} acc={m['accuracy']:.4f} (val_f1={best_val_f1:.4f}, {secs}s)")

        probs = softmax_np(test_logits)
        pdf = pd.DataFrame({"text": te["text"].values, "gold_label": te[lc].values, "pred_label": preds})
        for k in range(num_labels): pdf[f"prob_{k}"] = probs[:, k]
        pdf["encoder"] = etag; pdf["task"] = task
        pdf.to_csv(PRED / f"04_{etag}_{task}_test_predictions.csv", index=False)

        cm = confusion_matrix(te[lc].values, preds).tolist()
        confusions[f"{etag}_{task}"] = cm

        row = {"encoder": etag, "encoder_name": ename, "is_substitute": einfo["is_substitute"],
               "task": task, "num_labels": num_labels, "approach": "frozen_7dense_head",
               "pooling": POOLING, "head_hidden": json.dumps(HEAD_HIDDEN_UNITS), "head_dense_layers": 8,
               "in_dim": in_dim, "n_train": len(tr), "n_test": len(te),
               "best_val_macro_f1": best_val_f1, "epochs_ran": ep_ran, "time_seconds": secs,
               "leak_train_test": leak["train_test"]}
        row.update({f"test_{k}": v for k, v in m.items()})
        ref = REFERENCE_TARGETS.get((etag, task))
        if ref:
            row["reference_f1"] = ref.get("reference_f1")
            row["gap_vs_reference_f1"] = m["macro_f1"] - ref.get("reference_f1", 0.0)
        results.append(row)
        pd.DataFrame(results).to_csv(SUMMARY, index=False)

print("\nALL FROZEN-TRANSFER RUNS DONE")

resuming, done: 2



TASK: ben_sarc_binary  (labels=2, leak={'train_val': 0, 'train_test': 0, 'val_test': 0})
  skip sagorsarker-bb x ben_sarc_binary
  skip mbert x ben_sarc_binary

[bengali-bert] l3cube-pune/bengali-bert  -> extracting frozen embeddings ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at l3cube-pune/bengali-bert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    head: 30 epochs | best_val_f1=0.7514 | 59.3s
   -> test macro-F1=0.7444 acc=0.7444 (val_f1=0.7514, 76.5s)

TASK: banglasarc_binary  (labels=2, leak={'train_val': 0, 'train_test': 0, 'val_test': 0})

[sagorsarker-bb] sagorsarker/bangla-bert-base  -> extracting frozen embeddings ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

    head: 30 epochs | best_val_f1=0.6965 | 18.8s
   -> test macro-F1=0.7028 acc=0.7029 (val_f1=0.6965, 25.5s)

[mbert] bert-base-multilingual-cased  -> extracting frozen embeddings ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


    head: 30 epochs | best_val_f1=0.7013 | 21.0s
   -> test macro-F1=0.6898 acc=0.6928 (val_f1=0.7013, 28.2s)

[bengali-bert] l3cube-pune/bengali-bert  -> extracting frozen embeddings ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at l3cube-pune/bengali-bert and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    head: 30 epochs | best_val_f1=0.7087 | 21.7s
   -> test macro-F1=0.7213 acc=0.7244 (val_f1=0.7087, 28.7s)

ALL FROZEN-TRANSFER RUNS DONE


In [9]:
res = pd.read_csv(SUMMARY)
show = ["encoder","task","is_substitute","test_accuracy","test_macro_f1","best_val_macro_f1","time_seconds"]
print(res[show].to_string(index=False))

json.dump(confusions, open(TABLES / "04_confusion_matrices.json","w"), indent=2)

# frozen (this notebook) vs vanilla fine-tuned (NB06 multi-backbone), matched by encoder tag
ft = TABLES / "06_multi_backbone_summary.csv"
if ft.exists():
    ftdf = pd.read_csv(ft)[["backbone","task","test_macro_f1"]].rename(
        columns={"backbone":"encoder","test_macro_f1":"finetuned_macro_f1"})
    fr = res[["encoder","task","test_macro_f1"]].rename(columns={"test_macro_f1":"frozen_macro_f1"})
    comp = fr.merge(ftdf, on=["encoder","task"], how="left")
    comp["finetune_gain"] = comp["finetuned_macro_f1"] - comp["frozen_macro_f1"]
    comp.to_csv(TABLES / "04_frozen_vs_finetuned.csv", index=False)
    print("\nFROZEN (Table-10 head, this notebook) vs FINE-TUNED (NB06), matched by encoder tag:")
    print(comp.round(4).to_string(index=False))
else:
    print("\n(NB06 multi-backbone results not found yet; frozen-vs-finetuned table will fill in after NB06 runs.)")

       encoder               task  is_substitute  test_accuracy  test_macro_f1  best_val_macro_f1  time_seconds
sagorsarker-bb    ben_sarc_binary          False       0.692548       0.692125           0.696241       11248.2
         mbert    ben_sarc_binary          False       0.691767       0.691593           0.690767          87.3
  bengali-bert    ben_sarc_binary           True       0.744440       0.744425           0.751355          76.5
sagorsarker-bb  banglasarc_binary          False       0.922414       0.914145           0.910301          21.4
         mbert  banglasarc_binary          False       0.907328       0.899301           0.925262          19.3
  bengali-bert  banglasarc_binary           True       0.963362       0.959974           0.962436          15.9
sagorsarker-bb banglasarc3_binary          False       0.702908       0.702754           0.696505          25.5
         mbert banglasarc3_binary          False       0.692794       0.689777           0.701342       

In [10]:
# reference gap (only if REFERENCE_TARGETS filled) + file check
if "gap_vs_reference_f1" in res.columns and res["gap_vs_reference_f1"].notna().any():
    g = res.dropna(subset=["gap_vs_reference_f1"])[["encoder","task","test_macro_f1","reference_f1","gap_vs_reference_f1"]]
    g.to_csv(TABLES / "04_reference_gap.csv", index=False)
    print("Reference gap (Table 9, Ben-Sarc binary):\n", g.round(4).to_string(index=False))
else:
    print("REFERENCE_TARGETS empty — fill it from the Ben-Sarc paper to produce 04_reference_gap.csv")

print("\nFiles written:")
for d,prefix in [(TABLES,"04_"),(PRED,"04_"),(REPORTS,"04_")]:
    for p in sorted(d.glob(f"{prefix}*")):
        print("  ", p.relative_to(ROOT))

Reference gap (Table 9, Ben-Sarc binary):
        encoder            task  test_macro_f1  reference_f1  gap_vs_reference_f1
sagorsarker-bb ben_sarc_binary         0.6921        0.6641               0.0280
         mbert ben_sarc_binary         0.6916        0.6547               0.0369
  bengali-bert ben_sarc_binary         0.7444        0.7547              -0.0103

Files written:
   04_outputs/tables/04_confusion_matrices.json
   04_outputs/tables/04_frozen_transfer_results.csv
   04_outputs/tables/04_reference_gap.csv
   04_outputs/predictions/04_bengali-bert_banglasarc3_binary_test_predictions.csv
   04_outputs/predictions/04_bengali-bert_banglasarc_binary_test_predictions.csv
   04_outputs/predictions/04_bengali-bert_ben_sarc_binary_test_predictions.csv
   04_outputs/predictions/04_mbert_banglasarc3_binary_test_predictions.csv
   04_outputs/predictions/04_mbert_banglasarc_binary_test_predictions.csv
   04_outputs/predictions/04_mbert_ben_sarc_binary_test_predictions.csv
   04_output